# Pertemuan 5 — M2: Data Cleaning dan Justifikasi
**Tim:** Ajojing  
**Dataset:** SuperMarket Analysis.csv (Supermarket Sales)  
**Tujuan:** Menjalankan pipeline pembersihan yang dapat direproduksi, mencatat keputusan beserta alasannya, lalu memverifikasi kualitas data setelah dibersihkan.

> Catatan konsistensi proyek: pertanyaan payung pada notebook sebelumnya menyebut “Toko Buah Segar Utama” dan kolom `Total`, sedangkan dataset yang tersedia adalah transaksi supermarket dengan kolom `Sales`. Sebelum M3, tim perlu menyelaraskan pertanyaan analisis dengan dataset yang benar. Notebook ini tidak mengubah nama atau makna kolom secara diam-diam.


## Pertanyaan analisis proyek yang diselaraskan

**Pertanyaan payung:** Bagaimana pola penjualan dan pendapatan kotor pada transaksi supermarket berdasarkan waktu, cabang, kategori produk, tipe pelanggan, dan metode pembayaran selama periode Januari–Maret 2019?

**Pertanyaan turunan:**

- **1. Tren penjualan berdasarkan waktu:** Bagaimana perubahan total penjualan (Sales) harian dan bulanan selama periode Januari–Maret 2019? *(Kolom: `Date, Sales`)*
- **2. Perbandingan cabang:** Bagaimana perbandingan total penjualan (Sales) dan pendapatan kotor (gross income) pada setiap cabang selama periode pengamatan? *(Kolom: `Branch, City, Sales, gross income`)*
- **3. Kontribusi kategori produk:** Kategori produk (Product line) apa yang memberikan kontribusi terbesar terhadap total penjualan dan pendapatan kotor? *(Kolom: `Product line, Sales, gross income`)*
- **4. Pola transaksi menurut pelanggan:** Bagaimana perbandingan jumlah barang terjual (Quantity) berdasarkan tipe pelanggan (Customer type) dan gender? *(Kolom: `Quantity, Customer type, Gender`)*
- **5. Preferensi metode pembayaran:** Metode pembayaran (Payment) apa yang paling sering digunakan, dan bagaimana total penjualannya dibandingkan antarmetode pembayaran? *(Kolom: `Payment, Sales`)*

Rentang tanggal pada dataset yang tersedia adalah 1 Januari–9 Maret 2019 berdasarkan format tanggal yang digunakan. Karena itu, analisis tidak boleh menyatakan bahwa data mencakup seluruh Maret 2019.


## Anggota tim dan pembagian peran

- **Shafhan Farizi (0110223019) — Koordinator dan penanggung jawab analisis:** Menyelaraskan pertanyaan analisis, membagi pekerjaan, memeriksa konsistensi hasil, dan mengoordinasikan pemeriksaan akhir.
- **Abdul Aziz Alfarizi (0110225057) — Penanggung jawab data dan provenance:** Memeriksa sumber dataset, lisensi, tanggal pengambilan, kamus data, dan kesesuaian nama kolom.
- **Ziqri Akbar Ashari (0110225120) — Penanggung jawab notebook dan data cleaning:** Mengembangkan serta menjalankan pipeline Python, memeriksa kualitas data, dan menyimpan data hasil pembersihan.
- **Muhammad Rijal Jiddan (0110221246) — Penanggung jawab analisis, visualisasi, laporan, dan privasi:** Menghitung hasil pertanyaan turunan, menyiapkan tabel/grafik, menyusun laporan dan cleaning log, serta memeriksa penanganan ID dan repositori.


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# Cari CSV di folder notebook atau di data/raw/
candidate_paths = [
    Path("SuperMarket Analysis.csv"),
    Path("data/raw/SuperMarket Analysis.csv"),
    Path("../SuperMarket Analysis.csv"),
]
CSV_PATH = next((p for p in candidate_paths if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "Letakkan 'SuperMarket Analysis.csv' di folder notebook atau data/raw/."
    )

df_raw = pd.read_csv(CSV_PATH)
print("Sumber:", CSV_PATH)
print("Dimensi awal:", df_raw.shape)
display(df_raw.head())


In [ ]:
# Diagnosis awal: dimensi, tipe data, missing value, duplikat
print("Tipe data awal:")
display(df_raw.dtypes.rename("dtype").to_frame())

profil_awal = pd.DataFrame({
    "tipe_data": df_raw.dtypes.astype(str),
    "nilai_kosong": df_raw.isna().sum(),
    "persen_kosong": (df_raw.isna().mean() * 100).round(2),
    "nilai_unik": df_raw.nunique(dropna=True),
})
display(profil_awal)

print("Total sel kosong:", int(df_raw.isna().sum().sum()))
print("Duplikat persis:", int(df_raw.duplicated().sum()))
print("Invoice ID berulang:", int(df_raw["Invoice ID"].duplicated().sum()))


In [ ]:
# Pemeriksaan placeholder dan spasi pada teks
placeholder = {"", "na", "n/a", "null", "none", "unknown", "-"}
hasil_placeholder = {}
hasil_spasi = {}
for col in df_raw.columns:
    values = df_raw[col].astype(str)
    hasil_placeholder[col] = int(values.str.strip().str.lower().isin(placeholder).sum())
    if df_raw[col].dtype == "object":
        hasil_spasi[col] = int((values != values.str.strip()).sum())

print("Placeholder yang terdeteksi:", {k:v for k,v in hasil_placeholder.items() if v > 0})
print("Nilai teks dengan spasi tepi:", {k:v for k,v in hasil_spasi.items() if v > 0})


In [ ]:
# Pemeriksaan validitas dan hubungan antarkolom sebelum pembersihan
aturan_rentang = {
    "Unit price > 0": df_raw["Unit price"] > 0,
    "Quantity bilangan bulat dan >= 1": (df_raw["Quantity"] >= 1) & (df_raw["Quantity"] % 1 == 0),
    "Rating berada pada skala 0–10": df_raw["Rating"].between(0, 10),
    "Tax 5% tidak negatif": df_raw["Tax 5%"] >= 0,
    "Sales tidak negatif": df_raw["Sales"] >= 0,
    "cogs tidak negatif": df_raw["cogs"] >= 0,
    "gross income tidak negatif": df_raw["gross income"] >= 0,
}
for nama, valid in aturan_rentang.items():
    print(f"{nama}: {int((~valid).sum())} pelanggaran")

aturan_hubungan = {
    "cogs = Unit price * Quantity": np.isclose(df_raw["cogs"], df_raw["Unit price"] * df_raw["Quantity"]),
    "Tax 5% = 5% * cogs": np.isclose(df_raw["Tax 5%"], df_raw["cogs"] * 0.05),
    "Sales = cogs + Tax 5%": np.isclose(df_raw["Sales"], df_raw["cogs"] + df_raw["Tax 5%"]),
    "gross income = Tax 5%": np.isclose(df_raw["gross income"], df_raw["Tax 5%"]),
}
for nama, valid in aturan_hubungan.items():
    print(f"{nama}: {int((~valid).sum())} pelanggaran")


## Keputusan pembersihan

1. **Tipe data:** `Date` dan `Time` diubah menjadi tipe tanggal/waktu karena maknanya temporal dan diperlukan untuk analisis tren.
2. **Spasi teks:** gunakan `strip()` sebagai tindakan aman; pemeriksaan awal tidak menemukan nilai yang berubah.
3. **Placeholder:** tidak ada placeholder dari daftar pemeriksaan yang terdeteksi, jadi tidak ada nilai yang diubah menjadi kosong.
4. **Kategori:** kategori dipertahankan karena nilai kategori tampak konsisten; tidak ada dasar untuk menggabungkan kategori.
5. **Duplikat:** tidak menghapus baris karena tidak ditemukan duplikat persis maupun `Invoice ID` berulang.
6. **Nilai mustahil:** tidak mengubah nilai numerik karena aturan rentang yang diperiksa tidak menemukan pelanggaran.
7. **Nilai kosong:** tidak melakukan imputasi karena tidak ditemukan missing value.
8. **Kolom turunan:** menambahkan `year_month` dari `Date` untuk mendukung analisis bulanan.
9. **Pengenal transaksi:** `Invoice ID` dipseudonimkan sesuai keputusan kamus data sebelumnya. Hash bukan jaminan anonimitas mutlak; data tetap perlu dibagikan secara terbatas.


In [ ]:
def clean_supermarket_data(data: pd.DataFrame) -> pd.DataFrame:
    \"\"\"Bersihkan dataset supermarket secara deterministik dan dapat dijalankan ulang.\"\"\"
    d = data.copy()

    # 1. Rapikan spasi pada teks tanpa mengubah isi kategori.
    for col in d.select_dtypes(include="object").columns:
        d[col] = d[col].astype(str).str.strip()

    # 2. Konversi tanggal/waktu dengan format yang sesuai data sumber.
    d["Date"] = pd.to_datetime(d["Date"], format="%m/%d/%Y", errors="coerce")
    d["Time"] = pd.to_datetime(d["Time"], format="%I:%M:%S %p", errors="coerce")

    # 3. Pseudonimkan ID transaksi; awalan proyek hanya untuk memisahkan namespace hash.
    d["Invoice ID"] = d["Invoice ID"].astype(str).map(
        lambda x: hashlib.sha256(("ajojing-m2|" + x).encode("utf-8")).hexdigest()[:16]
    )

    # 4. Kolom turunan dibuat setelah tipe tanggal valid.
    d["year_month"] = d["Date"].dt.to_period("M").astype(str)

    return d

df_clean = clean_supermarket_data(df_raw)
print("Dimensi bersih:", df_clean.shape)
display(df_clean.head())


In [ ]:
# Verifikasi setelah pembersihan
ringkasan = pd.DataFrame([
    ["Jumlah baris", len(df_raw), len(df_clean)],
    ["Jumlah kolom", df_raw.shape[1], df_clean.shape[1]],
    ["Total sel kosong", int(df_raw.isna().sum().sum()), int(df_clean.isna().sum().sum())],
    ["Baris duplikat persis", int(df_raw.duplicated().sum()), int(df_clean.duplicated().sum())],
    ["Kategori Product line", int(df_raw["Product line"].nunique()), int(df_clean["Product line"].nunique())],
    ["ID transaksi unik", int(df_raw["Invoice ID"].nunique()), int(df_clean["Invoice ID"].nunique())],
], columns=["Ukuran", "Sebelum", "Sesudah"])
display(ringkasan)

print("Tipe Date sesudah:", df_clean["Date"].dtype)
print("Tipe Time sesudah:", df_clean["Time"].dtype)
print("Tanggal gagal dikonversi:", int(df_clean["Date"].isna().sum()))
print("Waktu gagal dikonversi:", int(df_clean["Time"].isna().sum()))


In [ ]:
# Dampak terhadap sebaran kolom numerik: Rating tidak diubah karena tidak ada pelanggaran
sebaran = pd.DataFrame({
    "Sebelum": df_raw["Rating"].agg(["mean", "median", "std"]),
    "Sesudah": df_clean["Rating"].agg(["mean", "median", "std"]),
})
sebaran["Perubahan"] = sebaran["Sesudah"] - sebaran["Sebelum"]
display(sebaran.round(4))
print("Interpretasi: statistik Rating tidak berubah karena nilai Rating tidak dimodifikasi.")


In [ ]:
# Simpan data hasil pembersihan dan profil diagnosis
out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)
reports_dir = Path("reports")
reports_dir.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(out_dir / "supermarket_clean.csv", index=False)
ringkasan.to_csv(reports_dir / "dampak_pembersihan.csv", index=False)
profil_awal.to_csv(reports_dir / "profil_kualitas_awal.csv")
print("Tersimpan:", out_dir / "supermarket_clean.csv")
print("Tersimpan:", reports_dir / "dampak_pembersihan.csv")
print("Tersimpan:", reports_dir / "profil_kualitas_awal.csv")


## Keterbatasan dan tindak lanjut

- Dataset tidak menyertakan informasi tentang mekanisme pengumpulan yang dapat menjelaskan potensi bias; MCAR/MAR/MNAR tidak relevan untuk kolom tanpa nilai kosong teramati, tetapi tidak adanya missing value tidak membuktikan data bebas bias.
- Format tanggal diasumsikan `bulan/hari/tahun` (`%m/%d/%Y`) sesuai format nilai sumber. Konfirmasi dokumentasi dataset tetap disarankan.
- Pertanyaan payung proyek di notebook Pertemuan 3 perlu diselaraskan: dataset ini berisi supermarket sales, bukan data khusus toko buah; kolom pendapatan penjualan yang tersedia bernama `Sales`, bukan `Total`.
- Jalankan **Restart Kernel and Run All** sebelum dikumpulkan. Jangan unggah data mentah ke repositori publik jika kebijakan tim/dosen melarangnya.
